# Modélisation — 4 Algorithmes (Classification Churn)

In [ ]:
import sys, os, time
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
from src.preprocessing import (
    preprocess_data, build_preprocessor, load_and_split,
    get_feature_names, compute_scale_pos_weight
)
from src.models.logistic_model import build_logistic_pipeline
from src.models.random_forest_model import build_rf_pipeline
from src.models.xgboost_model import build_xgb_pipeline
from src.models.mlp_model import train_mlp
from src.evaluate import compute_classification_metrics

In [ ]:
X_train, X_test, y_train, y_test, _ = load_and_split('../data/raw/customer_churn_business_dataset.csv')
preprocessor = build_preprocessor()
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
scale_pos_weight = compute_scale_pos_weight(y_train)
feature_names = get_feature_names(preprocessor)
print(f"scale_pos_weight: {scale_pos_weight:.2f} | Features: {len(feature_names)}")

## Modèle 1 — Régression Logistique (baseline)

In [ ]:
t0 = time.time()
lr_pipeline = build_logistic_pipeline(build_preprocessor())
lr_pipeline.fit(X_train, y_train)
print(f"Temps: {time.time()-t0:.1f}s")
y_pred_lr = lr_pipeline.predict(X_test)
y_proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]
metrics_lr = compute_classification_metrics('Logistic Regression', y_test, y_pred_lr, y_proba_lr)
print(metrics_lr)

## Modèle 2 — Random Forest

In [ ]:
rf_search = build_rf_pipeline(build_preprocessor())
t0 = time.time()
rf_search.fit(X_train, y_train)
print(f"Meilleurs params: {rf_search.best_params_}")
print(f"Temps: {time.time()-t0:.1f}s")
best_rf = rf_search.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]
metrics_rf = compute_classification_metrics('Random Forest', y_test, y_pred_rf, y_proba_rf)
print(metrics_rf)

## Modèle 3 — XGBoost

In [ ]:
xgb_search = build_xgb_pipeline(build_preprocessor(), scale_pos_weight=scale_pos_weight)
t0 = time.time()
xgb_search.fit(X_train, y_train)
print(f"Meilleurs params: {xgb_search.best_params_}")
print(f"Temps: {time.time()-t0:.1f}s")
best_xgb = xgb_search.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]
metrics_xgb = compute_classification_metrics('XGBoost', y_test, y_pred_xgb, y_proba_xgb)
print(metrics_xgb)

## Modèle 4 — MLP Deep Learning (Classification)

In [ ]:
mlp_model, mlp_history = train_mlp(X_train_proc, y_train.values, epochs=100, batch_size=256)
y_proba_mlp = mlp_model.predict(X_test_proc).flatten()
y_pred_mlp = (y_proba_mlp >= 0.5).astype(int)
metrics_mlp = compute_classification_metrics('MLP', y_test, y_pred_mlp, y_proba_mlp)
print(metrics_mlp)

## Modèle 5 — MLP Deep Learning (Régression CLV — BONUS)

In [ ]:
from src.preprocessing import load_and_split, build_preprocessor_regression, get_feature_names
from src.models.mlp_regressor_model import train_mlp_regressor
from src.evaluate import compute_regression_metrics

X_tr_r, X_te_r, y_tr_r, y_te_r, _ = load_and_split('../data/raw/customer_churn_business_dataset.csv', task='regression')
prep_r = build_preprocessor_regression()
X_tr_r_proc = prep_r.fit_transform(X_tr_r)
X_te_r_proc = prep_r.transform(X_te_r)

mlp_reg, history_reg = train_mlp_regressor(X_tr_r_proc, y_tr_r.values, epochs=100, batch_size=256)
y_pred_reg = mlp_reg.predict(X_te_r_proc).flatten()
metrics_reg = compute_regression_metrics('MLP Regressor', y_te_r, y_pred_reg)
print(metrics_reg)